# Tracing with MLflow

This notebook demonstrates how to use MLflow's tracing SDK to instrument your agent for MLflow. This allows you to track the execution of your agent and the metrics that are generated during its execution.

## Prerequisites

Complete the [Quick Start](../readme.md#quick-start) setup before running this notebook.

You will also need:
- Access to an OpenAI-compatible LLM endpoint (e.g. OpenAI, vLLM, llama.cpp, Ollama, etc.)

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# Load environment variables from .env file (searches current and parent directories)
load_dotenv(find_dotenv())

## E.g. For Ollama, uncomment and set these:
# os.environ["OPENAI_API_KEY"] = "EMPTY"
# os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"
# os.environ["OPENAI_MODEL_NAME"] = "gpt-oss:20b"

# Check that required vars are set, raise an error if not
required_vars = ["OPENAI_API_KEY"]
missing_vars = [var for var in required_vars if not os.getenv(var)]
if missing_vars:
    raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

## Start the Local MLflow Server

Now open a terminal in the parent directory and run the following command to start a local MLflow server

> ```sh
> uv run mlflow server --port 5001
> ```

Your server will be available on [http://localhost:5001](http://localhost:5001). Please open your browser and go to this URL to confirm the server is running.

## Environment Set Up

We need to set some environment variables to let MLflow know where to send traces. By default it will send traces to your local server.

We will start using the local server, but later we will show how to configure these to connect to a remote server in Openshift AI.

Note that these variables will take precedence over the variables you've set in your `.env` file.

In [ ]:
import os

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5001"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "Default"  # Use experiment name (works in both local and RHOAI)
os.environ["MLFLOW_TRACKING_AUTH"] = ""  # Tracking auth must not be set for local server

## The Basic Trace

Let's start by creating a simple function that we will trace.

Please note that there is a much simpler approach for tooling AI agents that we will demonstrate later in this notebook.

MLflow uses the `@mlflow.trace()` decorator to trace functions. This decorator will add a trace to the function that can be used to observe and visualize the function's execution.

In [ ]:
import mlflow

# A basic function with tracing enabled
@mlflow.trace()
def echo(input):
    # Simulate a prediction
    return f"You said: '{input}'"

Now we can test the function and see how the traces look in the UI:

In [ ]:
input = "Hello!"
res = echo(input)
print(res)

As you can see above, when working in Jupyter notebooks you'll see the trace appear directly below any cell that uses a traced function.
This gives you a quick way to view the whole trace from your development environment.

To view this same trace in the MLflow UI, navigate to Experiments > Default > Traces. 
Here you can see all traces that come into your experiment.

<center>
<img src="./images/basic_trace_dash.png" alt="drawing" width="75%"/>
<br>
<em>Traces dashboard</em>
</center>


Click on the most recent trace (top of the list) to see the details of the trace.


<center>
<img src="./images/basic_trace_view.png" alt="drawing" width="75%"/>
<br>
<em>Trace Details</em>
</center>

## Complex Traces

MLflow tracing can be customized in a number of ways:

- Add names to your traces so you can easily identify them in the UI.
- Add tags to your traces to categorize them.
- Add metadata to your traces to provide additional information about the trace.
- Nest your traces so you can see inputs and outputs for each step.

In [ ]:
import mlflow
import random

seed = random.randint(0, 100000)
rng = random.Random(seed)

# Give the trace a name and some attributes
@mlflow.trace(name="numbers", attributes={"is_random": "true", "seed": seed})
def get_numbers():
    a = rng.randint(1, 10)
    b = rng.randint(1, 100)
    
    # Add tags to the trace
    if a > 5 and b > 50:
        mlflow.update_current_trace(tags={"note": "It's a big one!"})
    else:
        mlflow.update_current_trace(tags={"note": "It's a small one..."})
        
    # Return a structured output that will render in UI
    return {"a": a, "b": b}


@mlflow.trace(name="multiply")
def multiply_numbers(a, b):
    result = a * b
    return result


@mlflow.trace(name="main_process")
def main(user):
    # Add a special tag to the trace to indicate the user
    mlflow.update_current_trace(metadata={"mlflow.trace.user": user})
    
    # Run other traced functions nested in another trace
    numbers = get_numbers()
    result = multiply_numbers(numbers["a"], numbers["b"])
    return f"The result of {numbers['a']} * {numbers['b']} is: {result}"

In [ ]:
user = "John Smith"
res = main(user)
print(res)

Open the trace and click on "Details & Timeline".
You can look at the individual traces of each function called and view their inputs, outputs, and attributes.

Near the ID of the trace you'll also see the user and tags we set in the functions.

<center>
<img src="./images/complex_trace_timeline.png" alt="drawing" width="75%"/>
<br>
<em>Trace Timeline</em>
</center>

## Instrumenting Agents with Autologgers

MLflow offers several out-of-the-box integrations with popular agent frameworks, including

- OpenAI Agents
- Pydantic AI
- LangChain / LangGraph
- Smolagents
- [And more...](https://mlflow.org/docs/latest/genai/tracing/integrations/)

Each of these only requires a single line of code to instrument


> ```py
> import mlflow
>
> mlflow.<framework>.autolog()
> ```


In this section, we'll demonstrate how to trace an OpenAI Agent using the autologger.

In [ ]:
import mlflow

# Now enable MLflow tracing - Just one line!
mlflow.openai.autolog()

In [ ]:
from agents import Agent, Runner, set_default_openai_client
from openai import AsyncClient
import os

model = os.getenv("OPENAI_MODEL_NAME", "gpt-4o")

# Define a simple multi-agent workflow
spanish_agent = Agent(
    name="Spanish agent",
    instructions="You only speak Spanish.",
    model=model,
)

english_agent = Agent(
    name="English agent",
    instructions="You only speak English",
    model=model,
)

triage_agent = Agent(
    name="Triage agent",
    instructions="Handoff to the appropriate agent based on the language of the request.",
    handoffs=[spanish_agent, english_agent],
    model=model,
)


async def invoke(input):
    # Configure OpenAI-compatible endpoint
    async_client = AsyncClient(
        base_url=os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
        api_key=os.environ.get("OPENAI_API_KEY", ""),
    )
    set_default_openai_client(client=async_client)
    
    # Run agent
    result = await Runner.run(triage_agent, input=input)
    return result.final_output

In [ ]:
input = "Hola, ¿cómo estás?"
res = await invoke(input)
print(res)

Open the trace and click on "Details & Timeline".

You can clearly see the triage agent was called, then there was a handoff to the spanish agent, and finally a response to the user.

Near the trace ID you can also see the total token usage for your request and response.

<center>
<img src="./images/agent_timeline.png" alt="drawing" width="75%"/>
<br>
<em>Agent Timeline</em>
</center>

## Configure Tracing to Openshift AI MLflow Tracking Server

To configure local MLflow traces to be sent to an RHOAI MLflow Tracking Server, you first need to sign into the Openshift cluster.
Copy the login command from your cluster and log in to your cluster.

Once logged in, set your project to match your experiment with `oc project <your-project>`.

Then, you can configure MLflow to send traces to the RHOAI MLflow Tracking Server by setting the following environment variables:

In [ ]:
import os

os.environ["MLFLOW_TRACKING_URI"]='https://<RHOAI Cluster URL>/mlflow'
os.environ["MLFLOW_TRACKING_AUTH"]='kubernetes'
os.environ["MLFLOW_EXPERIMENT_NAME"]='nps-agent'  # Use experiment name (works in both local and RHOAI)

Please note that this only works in `mlflow>=3.10.0`. For prior versions, please install the latest Red Hat build of mlflow:

> ```sh
> uv pip install "git+https://github.com/opendatahub-io/mlflow@master"
> ```

Now you can rerun the above examples and see the traces logged in the RHOAI MLflow Tracking Server.
We'll demonstrate this in the next notebook as well.